In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb

pd.set_option('display.max_columns', 10)

In [14]:
# Load data to df
df = pd.read_csv('./data/german_credit_data_updated.csv')

# Drop ID column
df = df.drop('ID', axis=1)

encoder = OneHotEncoder(sparse_output=False, drop='first', dtype=int)
encoded_cols = encoder.fit_transform(df[['Sex', 'Purpose']])
encoded_feature_names = encoder.get_feature_names_out(['Sex', 'Purpose'])
encoded_df = pd.DataFrame(encoded_cols, columns=encoded_feature_names, index=df.index).astype(int)
df = df.drop(['Sex', 'Purpose'], axis=1)
df = pd.concat([df, encoded_df], axis=1)

# df['Sex'] = df['Sex'].astype('category')
# df['Purpose'] = df['Purpose'].astype('category')

# Ordinal mappings for housing status
housingNumericMap = {
        "free": 0,
        "rent": 1,
        "own": 2
}

# Remap checking account status to ordinal
df['Housing'] = df['Housing'].map(housingNumericMap)

# Ordinal mappings for account status
accountNumericMap = {
        "little": 1,
        "moderate": 2,
        "quite rich": 3,
        "rich": 4
}

# Remap saving account status to ordinal
df['Saving accounts'] = df['Saving accounts'].map(accountNumericMap)
df['Saving accounts'] = df['Saving accounts'].fillna(0).astype(int)

# Remap checking account status to ordinal
df['Checking account'] = df['Checking account'].map(accountNumericMap)
df['Checking account'] = df['Checking account'].fillna(0).astype(int)

df['Credit Risk'] = df['Credit Risk'] - 1
# df = df.drop("Sex", axis=1)
# df = df.drop("Purpose", axis=1)
cols = ["Credit Risk"] + [c for c in df.columns if c != "Credit Risk"]
df = df[cols]
df.to_csv("german_credit_data_onehot.csv", index=False)
df.head()

,Credit Risk,Age,Job,Housing,Saving accounts,...,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others
0,0,67,2,2,0,...,0,0,1,0,0
1,1,22,2,2,1,...,0,0,1,0,0
2,0,49,1,2,1,...,1,0,0,0,0
3,0,45,2,0,1,...,0,1,0,0,0
4,1,53,2,0,1,...,0,0,0,0,0


In [16]:
print(",".join([str(x) for x in df.drop(columns=['Credit Risk']).mean().values]))

35.501048218029354,1.909853249475891,1.6027253668763102,1.1907756813417192,1.0576519916142557,3279.1121593291405,20.78092243186583,0.6876310272536688,0.33752620545073375,0.012578616352201259,0.06079664570230608,0.18343815513626835,0.2746331236897275,0.020964360587002098,0.012578616352201259


In [10]:
X = df.drop('Credit Risk', axis=1)
y = df['Credit Risk']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,        # ~800:200 training test split
    random_state = 7777777, # Fixed random seed for reproducability
    stratify = y            # Ensure even proportion of rejected loans between sets
)

In [11]:
xgbModel = xgb.XGBClassifier(
    learning_rate=0.1,
    enable_categorical=True
)

xgbModel.fit(X_train, y_train)

y_pred = xgbModel.predict(X_test)

print(accuracy_score(y_test, y_pred))

0.7172774869109948
